# SMIRK on Kaggle: setup and demo

Installs SMIRK (code from github.com/THUONGVIP09/smirk, branch occlusion-aware), downloads FLAME 2020 with your FLAME
account and the pretrained SMIRK_em1.pt, then reconstructs some test faces of clean-faces.

Before running:
- Settings: **Accelerator GPU T4 x1**, **Internet on**.
- *Add-ons → Secrets*: add `FLAME_USERNAME` (FLAME account email) and `FLAME_PASSWORD`, and enable them for this notebook.
- Inputs: `clean-faces`, `affectnet-crops`, `expw-crops`. If a dataset with a `pytorch3d-*.whl` (output of a previous
  run of this notebook) is added as input, PyTorch3D is installed from it instead of being built (saves ~20-30 min).

FLAME and SMIRK may not be redistributed: keep this notebook and its outputs **Private**.

## 1. Environment

In [ ]:
import sys, os, glob, subprocess
import torch, numpy
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda,
      '| numpy', numpy.__version__, '| GPU', torch.cuda.is_available())
assert torch.cuda.is_available(), 'enable the GPU accelerator'

## 2. Get the SMIRK code

In [ ]:
import os
os.chdir('/kaggle/working')  # leave /tmp/smirk before deleting it (a later cell changes into it)
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1

## 3. Python packages

timm is pinned to the version SMIRK was trained with (the encoder weights are stored with its layer names).
PyTorch3D has no wheel for this Python / PyTorch, so it is built from source once (~20-30 min) and the wheel is kept in
the output, to be reused as an input dataset next time.

In [ ]:
!pip install -q timm==0.9.16 iopath gdown

wheels = glob.glob('/kaggle/input/**/pytorch3d-*.whl', recursive=True)
if wheels:
    print('installing', wheels[0])
    !pip install -q {wheels[0]}
else:
    os.environ.update(FORCE_CUDA='1', TORCH_CUDA_ARCH_LIST='7.5', MAX_JOBS='4')  # T4 = compute capability 7.5
    !pip wheel --no-build-isolation --no-deps -w /kaggle/working/wheels "git+https://github.com/facebookresearch/pytorch3d.git@main"
    !pip install -q /kaggle/working/wheels/pytorch3d-*.whl

import pytorch3d
print('pytorch3d', pytorch3d.__version__)

## 4. FLAME 2020 (downloaded with the account stored in Kaggle Secrets)

In [ ]:
from kaggle_secrets import UserSecretsClient

assert os.path.isdir('/tmp/smirk/assets'), 'the SMIRK code is missing in /tmp/smirk: run the cell that clones the repository first'
os.makedirs('/tmp/smirk/assets/FLAME2020', exist_ok=True)

import requests, time, zipfile

# kept in /tmp (not in the notebook output: FLAME may not be redistributed), so that re-running this cell
# in the same session does not download it again
FLAME_ZIP = '/tmp/FLAME2020.zip'
if not (os.path.exists(FLAME_ZIP) and zipfile.is_zipfile(FLAME_ZIP)):
    secrets = UserSecretsClient()
    credentials = {'username': secrets.get_secret('FLAME_USERNAME'), 'password': secrets.get_secret('FLAME_PASSWORD')}
    url = 'https://download.is.tue.mpg.de/download.php?domain=flame&sfile=FLAME2020.zip&resume=1'
    for attempt in range(3):  # the credentials are sent in the request body only, never printed
        try:
            with requests.post(url, data=credentials, stream=True, timeout=600) as response:
                status = response.status_code
                if status == 200:
                    with open(FLAME_ZIP, 'wb') as f:
                        for chunk in response.iter_content(1 << 20):
                            f.write(chunk)
        except requests.RequestException as e:
            status = type(e).__name__
        if os.path.exists(FLAME_ZIP) and zipfile.is_zipfile(FLAME_ZIP):
            break
        print(f'FLAME download attempt {attempt + 1} failed (HTTP status {status}), retrying in 30 s')
        time.sleep(30)
    del credentials
    assert zipfile.is_zipfile(FLAME_ZIP), ('FLAME download failed: HTTP 401/403 = wrong FLAME_USERNAME / FLAME_PASSWORD '
                                           'or account not activated; 429/5xx = server busy, try again later')

import shutil
assert subprocess.run(['unzip', '-q', '-o', FLAME_ZIP, '-d', '/tmp/smirk/assets/FLAME2020/']).returncode == 0, \
    'could not extract the FLAME zip'

# the zip may contain a FLAME2020/ folder: move its files to assets/FLAME2020/ where SMIRK looks for them
found = glob.glob('/tmp/smirk/assets/FLAME2020/**/generic_model.pkl', recursive=True)
assert found, 'generic_model.pkl is not in the FLAME zip'
source = os.path.dirname(found[0])
if source != '/tmp/smirk/assets/FLAME2020':
    for name in os.listdir(source):
        shutil.move(os.path.join(source, name), '/tmp/smirk/assets/FLAME2020/')
!ls -la /tmp/smirk/assets/FLAME2020/

## 5. Pretrained SMIRK (SMIRK_em1.pt, from the authors' Google Drive)

In [ ]:
!mkdir -p /tmp/smirk/pretrained_models
!gdown -q 1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE -O /tmp/smirk/pretrained_models/SMIRK_em1.pt
!ls -la /tmp/smirk/pretrained_models/

## 6. Load FLAME, renderer and encoder

In [ ]:
os.chdir('/tmp/smirk')
sys.path.insert(0, '/tmp/smirk')
from src.smirk_encoder import SmirkEncoder
from src.FLAME.FLAME import FLAME
from src.renderer.renderer import Renderer

DEVICE = 'cuda'
flame = FLAME().to(DEVICE)
print('FLAME shapedirs', tuple(flame.shapedirs.shape), '(expected (5023, 3, 350))')
renderer = Renderer().to(DEVICE)

encoder = SmirkEncoder().to(DEVICE).eval()
checkpoint = torch.load('pretrained_models/SMIRK_em1.pt', map_location=DEVICE)
encoder.load_state_dict({k.replace('smirk_encoder.', ''): v for k, v in checkpoint.items() if k.startswith('smirk_encoder')})
print('SMIRK encoder loaded')

## 7. Reconstruct some clean test faces (2 per expression)

In [ ]:
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt

clean_dir = os.path.dirname(glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True)[0])
meta = pd.read_csv(f'{clean_dir}/clean_meta.csv', dtype={'name': str})
landmark_data = np.load(f'{clean_dir}/landmarks.npz')
landmarks_of = dict(zip(landmark_data['keys'], landmark_data['landmarks']))

image_dirs = {}
for meta_csv in glob.glob('/kaggle/input/**/meta.csv', recursive=True):
    folder = os.path.dirname(meta_csv)
    # dataset of an image folder, from the keyword in its path (works for rafdb_crops, rafdb-crops, affectnet_1, ...)
    path = os.path.relpath(folder, '/kaggle/input').lower()
    name = next((k for k in ('expw', 'affectnet', 'rafdb') if k in path), None)
    if name and os.path.isdir(os.path.join(folder, 'images')):
        image_dirs[name] = os.path.join(folder, 'images')
print('image folders:', image_dirs)

def smirk_crop(image, landmarks, scale=1.4, size=224):
    # same crop as demo.py / BaseDataset.crop_face
    left, top = landmarks.min(axis=0); right, bottom = landmarks.max(axis=0)
    old_size = (right - left + bottom - top) / 2
    center = np.array([(left + right) / 2.0, (top + bottom) / 2.0])
    s = int(old_size * scale)
    src = np.float32([[center[0] - s / 2, center[1] - s / 2], [center[0] - s / 2, center[1] + s / 2], [center[0] + s / 2, center[1] - s / 2]])
    dst = np.float32([[0, 0], [0, size - 1], [size - 1, 0]])
    return cv2.warpAffine(image, cv2.getAffineTransform(src, dst), (size, size), borderValue=0)

test = meta[(meta['split'] == 'test') & meta['dataset'].isin(list(image_dirs))]
sample = test.groupby('expression', group_keys=False).apply(lambda g: g.sample(min(2, len(g)), random_state=0))
crops = [smirk_crop(cv2.imread(f"{image_dirs[r['dataset']]}/{r['name']}.png")[..., ::-1],
                    landmarks_of[f"{r['dataset']}/{r['name']}"][:, :2]) for _, r in sample.iterrows()]

images = torch.tensor(np.stack(crops)).permute(0, 3, 1, 2).float().div(255).to(DEVICE)
with torch.no_grad():
    outputs = encoder(images)
    rendered = renderer.forward(flame.forward(outputs)['vertices'], outputs['cam'])['rendered_img']

fig, axes = plt.subplots(3, len(crops), figsize=(2 * len(crops), 6.5))
for i, (_, r) in enumerate(sample.iterrows()):
    img, ren = crops[i] / 255.0, rendered[i].permute(1, 2, 0).cpu().numpy()
    for ax, picture in zip(axes[:, i], [img, ren, np.where(ren.sum(-1, keepdims=True) > 0, 0.6 * ren + 0.4 * img, img)]):
        ax.imshow(np.clip(picture, 0, 1)); ax.axis('off')
    axes[0, i].set_title(f"{r['dataset']}\n{r['expression']}", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Synthetic occlusions: SMIRK on clean vs occluded faces

Occludes random parts (mouth, eyes, eyebrows, nose) of the same test faces with random shapes, sizes and appearances
(src/utils/occlusion.py, fixed seed per image), then compares the SMIRK reconstructions of the clean and occluded images.
`d_exp` = distance between the expression parameters predicted from the clean and from the occluded image.

In [ ]:
from src.utils.occlusion import FaceOccluder, occlusion_rng

def crop_matrix(landmarks, scale=1.4, size=224):
    left, top = landmarks.min(axis=0); right, bottom = landmarks.max(axis=0)
    old_size = (right - left + bottom - top) / 2
    center = np.array([(left + right) / 2.0, (top + bottom) / 2.0])
    s = int(old_size * scale)
    src = np.float32([[center[0] - s / 2, center[1] - s / 2], [center[0] - s / 2, center[1] + s / 2], [center[0] + s / 2, center[1] - s / 2]])
    return cv2.getAffineTransform(src, np.float32([[0, 0], [0, size - 1], [size - 1, 0]]))

occluder = FaceOccluder()
clean, occluded, titles = [], [], []
for _, r in sample.iterrows():
    image = cv2.imread(f"{image_dirs[r['dataset']]}/{r['name']}.png")[..., ::-1]
    landmarks = landmarks_of[f"{r['dataset']}/{r['name']}"][:, :2]
    M = crop_matrix(landmarks)
    crop = cv2.warpAffine(image, M, (224, 224), borderValue=0)
    crop_landmarks = landmarks @ M[:, :2].T + M[:, 2]       # landmarks in crop coordinates
    occluded_crop, mask, info = occluder(crop, crop_landmarks, rng=occlusion_rng(f"{r['dataset']}/{r['name']}"))
    clean.append(crop); occluded.append(occluded_crop)
    titles.append(f"{r['expression']}\n{'+'.join(info['parts'])}")

def reconstruct(crops):
    x = torch.tensor(np.stack(crops)).permute(0, 3, 1, 2).float().div(255).to(DEVICE)
    with torch.no_grad():
        out = encoder(x)
        return out, renderer.forward(flame.forward(out)['vertices'], out['cam'])['rendered_img']

out_clean, ren_clean = reconstruct(clean)
out_occ, ren_occ = reconstruct(occluded)
d_exp = (out_occ['expression_params'] - out_clean['expression_params']).norm(dim=1).cpu().numpy()
print('mean expression distance clean vs occluded:', d_exp.mean().round(3))

rows = [('clean', clean), ('SMIRK clean', ren_clean), ('occluded', occluded), ('SMIRK occluded', ren_occ)]
fig, axes = plt.subplots(4, len(clean), figsize=(2 * len(clean), 9))
for row, (label, pictures) in zip(axes, rows):
    for i, ax in enumerate(row):
        p = pictures[i]
        p = p.permute(1, 2, 0).cpu().numpy() if torch.is_tensor(p) else p / 255.0
        ax.imshow(np.clip(p, 0, 1)); ax.axis('off')
    row[0].set_title(label, loc='left', fontsize=9)
for i, ax in enumerate(axes[2]):
    ax.set_title(f'{titles[i]}\nd_exp={d_exp[i]:.2f}', fontsize=7)
plt.tight_layout(); plt.show()

## 9. Save

*Save Version → Save & Run All*. The output keeps `wheels/pytorch3d-*.whl`: create a **Private** dataset from it
(for example `pytorch3d-wheel`) and add it as input next time to skip the PyTorch3D build.